# Terafab Decision Twin — Integrated Colab Lab

This notebook is a single, Google Colab-friendly lab for the source-available Terafab Decision Twin package. It replaces the previous notebook set and walks through public monitoring examples, deterministic scenario simulation, strategic simulation, and validation-readiness screening.

**Rights and trust boundary.** This repository is source-available, not open source. The examples in this notebook use public examples and declared assumptions only. Outputs are conditional analytical estimates, not verified Terafab operating facts, engineering certifications, investment advice, procurement guidance, permitting determinations, or evidence of Terafab endorsement. This independent model is not affiliated with, endorsed by, authorized by, sponsored by, or connected to Terafab or its employees unless expressly stated in writing by the relevant official entity.

## 1. Setup

Run this notebook from a cloned repository in Colab or from the repository root locally. The setup cell locates the package source and imports the public APIs used throughout the lab.

In [ ]:
from __future__ import annotations

import copy
import json
import math
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd()
if not (ROOT / "terafab_decision_twin").exists():
    candidates = [Path("/content/terafab-decision-twin"), Path("/workspace/terafab-decision-twin")]
    ROOT = next((p for p in candidates if (p / "terafab_decision_twin").exists()), ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from terafab_decision_twin.engine import MODEL_VERSION, run_scenario
from terafab_decision_twin.report import markdown_report
from terafab_decision_twin.schema import load_scenario, validate_scenario
from strategic_simulation import (
    analyze_normal_form_game,
    build_stakeholder_decision_surface,
    run_monte_carlo,
    simulate_reduced_order_model,
)
from validation_lab.validation_report import build_validation_report, load_reference_ranges

print(f"Repository root: {ROOT}")
print(f"Model version: {MODEL_VERSION}")

## 2. Public monitoring examples

The `examples/` directory provides public-monitoring templates, stakeholder workflows, and two schema-valid demonstration scenarios. The first step is to verify that those assets exist and to view their intended role.

In [ ]:
example_files = {
    "Checklist": ROOT / "examples/public_monitoring_checklist.md",
    "Evidence intake template": ROOT / "examples/evidence_intake_template.json",
    "Source register template": ROOT / "examples/source_register_template.json",
    "Board workflow": ROOT / "examples/stakeholder_board_monitoring.md",
    "Policy workflow": ROOT / "examples/stakeholder_policy_monitoring.md",
    "Engineering workflow": ROOT / "examples/stakeholder_engineering_monitoring.md",
    "Investor workflow": ROOT / "examples/stakeholder_investor_monitoring.md",
    "Progress scenario": ROOT / "examples/scenario_public_progress_watch.json",
    "Infrastructure scenario": ROOT / "examples/scenario_public_infrastructure_watch.json",
}
assets = pd.DataFrame(
    [{"asset": name, "path": str(path.relative_to(ROOT)), "exists": path.exists()} for name, path in example_files.items()]
)
display(assets)
assert assets["exists"].all(), "One or more public monitoring example files are missing."

In [ ]:
evidence_template = json.loads((ROOT / "examples/evidence_intake_template.json").read_text())
source_template = json.loads((ROOT / "examples/source_register_template.json").read_text())

print("Evidence intake template keys:", list(evidence_template.keys()))
print("Source register template keys:", list(source_template.keys()))
display(pd.DataFrame([{"template": "evidence_intake", "keys": ", ".join(evidence_template.keys())}, {"template": "source_register", "keys": ", ".join(source_template.keys())}]))

In [ ]:
public_scenarios = [
    ROOT / "examples/scenario_public_progress_watch.json",
    ROOT / "examples/scenario_public_infrastructure_watch.json",
]
validation_rows = []
for path in public_scenarios:
    scenario = load_scenario(path)
    errors = validate_scenario(scenario)
    validation_rows.append({"scenario": str(path.relative_to(ROOT)), "valid": not errors, "errors": errors})
public_validation_df = pd.DataFrame(validation_rows)
display(public_validation_df)
assert public_validation_df["valid"].all(), "A public example scenario failed schema validation."

## 3. Deterministic scenario simulation

This section runs the deterministic decision twin on the public progress scenario, summarizes gates and evidence status, and renders the built-in Markdown report. Passing validation or gates does not convert assumptions into verified operating facts.

In [ ]:
scenario_path = ROOT / "examples/scenario_public_progress_watch.json"
scenario = load_scenario(scenario_path)
result = run_scenario(scenario)

summary = result.get("summary", {})
print("Scenario:", scenario_path.relative_to(ROOT))
print("Overall gate result:", "PASS" if result.get("passed") else "FAIL")

key_metrics = [
    "recommended_phase_action",
    "minimum_firm_capacity_margin_MW",
    "minimum_heat_rejection_margin_MW",
    "minimum_water_withdrawal_margin_m3_per_day",
    "minimum_wastewater_discharge_margin_m3_per_day",
    "average_effective_yield",
    "average_readiness_index",
    "total_cost_USD",
    "cost_per_good_die_USD",
    "emissions_tCO2",
    "legitimacy_margin",
]
display(pd.DataFrame([{"metric": k, "value": summary.get(k)} for k in key_metrics if k in summary]))

In [ ]:
gates_df = pd.DataFrame(result.get("gates", []))
if not gates_df.empty:
    display(gates_df[["name", "passed", "severity", "margin", "message"]])
    gate_counts = gates_df["passed"].value_counts().rename(index={True: "pass", False: "fail"})
    ax = gate_counts.plot(kind="bar", title="Gate status counts")
    ax.set_ylabel("count")
    plt.show()

In [ ]:
evidence = result.get("evidence", {})
evidence_counts = evidence.get("status_counts", {})
evidence_df = pd.DataFrame([{"status": k, "count": v} for k, v in evidence_counts.items()])
display(evidence_df)
if not evidence_df.empty:
    ax = evidence_df.plot(x="status", y="count", kind="bar", legend=False, title="Evidence status counts")
    ax.set_ylabel("inputs")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

In [ ]:
display(Markdown(markdown_report(result)))

## 4. Strategic simulation examples

The strategic layer adds uncertainty propagation, declared-payoff game analysis, reduced-order trajectory simulation, and a stakeholder-facing decision surface. These are screening tools derived from declared inputs, not observed stakeholder preferences or official validation.

In [ ]:
mc_config_path = ROOT / "strategic_simulation/examples/uncertainty_baseline_2026.json"
mc_config = json.loads(mc_config_path.read_text())
base_scenario_path = ROOT / mc_config["base_scenario"]

# Keep the Colab default quick. Increase this value for deeper sampling.
mc_config_quick = copy.deepcopy(mc_config)
mc_config_quick["runs"] = min(int(mc_config_quick.get("runs", 50)), 40)

mc_result = run_monte_carlo(base_scenario_path, mc_config_quick, retain_runs=False)
print("Monte Carlo runs completed:", mc_result["runs_completed"])
print("Passed probability:", mc_result["passed_probability"])

gate_failure_df = pd.DataFrame(
    [{"gate": k, "failure_probability": v} for k, v in mc_result.get("gate_failure_probability", {}).items()]
).sort_values("failure_probability", ascending=False)
display(gate_failure_df)
if not gate_failure_df.empty:
    ax = gate_failure_df.plot(x="gate", y="failure_probability", kind="bar", legend=False, title="Gate failure probabilities")
    ax.set_ylabel("probability")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

In [ ]:
quantile_rows = []
for metric, qs in mc_result.get("metric_quantiles", {}).items():
    row = {"metric": metric}
    row.update(qs)
    quantile_rows.append(row)
quantiles_df = pd.DataFrame(quantile_rows)
display(quantiles_df.head(20))

sensitivity_df = pd.DataFrame(mc_result.get("sensitivity", []))
display(sensitivity_df.head(10))
if not sensitivity_df.empty:
    ax = sensitivity_df.head(8).plot(x="parameter", y="absolute_correlation", kind="bar", legend=False, title="Sensitivity ranking")
    ax.set_ylabel("absolute Pearson correlation")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

In [ ]:
game_config_path = ROOT / "strategic_simulation/examples/strategic_game_2026.json"
game_config = json.loads(game_config_path.read_text())
game_result = analyze_normal_form_game(game_config)

print("Actors:", game_result["actors"])
print("Profile count:", game_result["profile_count"])
print("Conflict index:", game_result["conflict_index"])
print("Coordination failure warning:", game_result["coordination_failure_warning"])

display(pd.DataFrame(game_result["pure_strategy_nash_equilibria"]))
display(pd.DataFrame(game_result["total_payoff_ranking"]))

In [ ]:
rom_config_path = ROOT / "strategic_simulation/examples/rom_transition_2026_2030.json"
rom_config = json.loads(rom_config_path.read_text())
rom_result = simulate_reduced_order_model(rom_config)
trajectory_df = pd.DataFrame(rom_result["trajectory"])
display(trajectory_df)

plot_df = trajectory_df.set_index("step")
ax = plot_df.plot(title="Reduced-order trajectory")
ax.set_ylabel("state index")
plt.show()
print("Diagnostics:", rom_result.get("diagnostics", {}))

In [ ]:
reference_ranges = load_reference_ranges(ROOT / "validation_lab/reference_ranges.json")
validation_report = build_validation_report(result, reference_ranges)
surface = build_stakeholder_decision_surface(
    monte_carlo=mc_result,
    game=game_result,
    rom=rom_result,
    validation=validation_report,
)
print("Recommended phase posture:", surface["recommended_phase_posture"])
print("Concerns:", surface["concerns"])
for view in ["board_view", "investor_view", "policy_view", "research_view"]:
    display(Markdown(f"### {view.replace('_', ' ').title()}"))
    display(pd.DataFrame([surface[view]]).T.rename(columns={0: "value"}))

In [ ]:
integrated_path = ROOT / "strategic_simulation/examples/integrated_advanced_2026_2030.json"
integrated = json.loads(integrated_path.read_text())
display(pd.DataFrame([{"field": k, "value": v} for k, v in integrated.items()]))

integrated_summary = {
    "base_scenario": integrated.get("base_scenario"),
    "monte_carlo_runs": mc_result.get("runs_completed"),
    "monte_carlo_failed_probability": mc_result.get("failed_probability"),
    "game_profiles": game_result.get("profile_count"),
    "game_conflict_index": game_result.get("conflict_index"),
    "rom_steps": rom_result.get("steps"),
    "stakeholder_posture": surface.get("recommended_phase_posture"),
}
display(pd.DataFrame([integrated_summary]).T.rename(columns={0: "value"}))

## 5. Validation lab

The validation lab performs structural and public-reference-range screening. It does not certify the model or validate official Terafab operating data.

In [ ]:
validation_case_path = ROOT / "validation_lab/examples/public_reference_validation_case.json"
validation_case = json.loads(validation_case_path.read_text())
reference_ranges = load_reference_ranges(ROOT / validation_case["reference_ranges"])
validation_report = build_validation_report(result, reference_ranges)

display(pd.DataFrame(validation_report["checks"]))
display(pd.DataFrame([validation_report["scorecard"]]))
range_flags = pd.DataFrame(validation_report.get("range_flags", []))
print("Validation level:", validation_report["validation_level"])
print("Calibration gaps:", validation_report["calibration_gaps"])
display(range_flags if not range_flags.empty else pd.DataFrame([{"range_flags": "none"}]))

## 6. End-to-end decision-support dashboard

This compact dashboard connects public monitoring, schema validation, deterministic simulation, strategic simulation, and validation-readiness screening. It should be read as decision-support context, not as verified fact or official guidance.

In [ ]:
failed_gates = [g["name"] for g in result.get("gates", []) if not g.get("passed")]
dashboard = {
    "scenario_id": result.get("metadata", {}).get("scenario_id"),
    "schema_valid": len(validate_scenario(scenario)) == 0,
    "deterministic_gate_result": "PASS" if result.get("passed") else "FAIL",
    "failed_gates": ", ".join(failed_gates) if failed_gates else "none",
    "evidence_status_counts": evidence_counts,
    "underdetermined_unknowns": result.get("unknowns", {}).get("underdetermined"),
    "mc_failed_probability": mc_result.get("failed_probability"),
    "game_conflict_index": game_result.get("conflict_index"),
    "rom_terminal_state": rom_result.get("terminal_state"),
    "stakeholder_posture": surface.get("recommended_phase_posture"),
    "validation_level": validation_report.get("validation_level"),
    "validation_range_flags": len(validation_report.get("range_flags", [])),
}
display(pd.DataFrame([dashboard]).T.rename(columns={0: "value"}))

next_data = result.get("interpretation", {}).get("highest_value_next_data", [])
if next_data:
    display(Markdown("### Highest-value next data"))
    display(pd.DataFrame({"next_data": next_data}))

## 7. Reproducibility notes

- Default Monte Carlo runs are capped for interactive Colab use.
- Increase the run count only after confirming the notebook runs end-to-end.
- Preserve source-available, non-affiliation, evidence-status, assumption-boundary, and scenario-dependence notices in derivative notebooks or reports.
- Passing notebook cells does not mean model outputs are verified Terafab facts.